In [1]:
import sys,os,time,re,json,torch,transformers
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM, GenerationConfig
from tqdm import tqdm
from PIL import Image

## 加载本地OCR模型

In [2]:
# 检查GPU是否可用
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
print("Current CUDA device:", torch.cuda.current_device())
print("Torch version:", torch.__version__)

print(sys.version)

torch.cuda.empty_cache()

CUDA available: True
CUDA device count: 8
Current CUDA device: 0
Torch version: 2.6.0+cu124
3.12.9 (main, Mar 17 2025, 21:01:58) [Clang 20.1.0 ]


In [3]:
# 检查当前notebook使用的Python环境
print("Python executable:", sys.executable)
print("Python version:", sys.version)

# 检查transformers版本和位置
print("\nTransformers version:", transformers.__version__)
print("Transformers location:", transformers.__file__)

Python executable: /data/home/yunhao/code/ocr/.venv/bin/python
Python version: 3.12.9 (main, Mar 17 2025, 21:01:58) [Clang 20.1.0 ]

Transformers version: 4.46.3
Transformers location: /data/home/yunhao/code/ocr/.venv/lib/python3.12/site-packages/transformers/__init__.py


In [15]:
ocr_model_path = "../model/deepseek-ocr"

tokenizer = AutoTokenizer.from_pretrained(ocr_model_path, _attn_implementation='flash_attention_2', trust_remote_code=True)
model = AutoModel.from_pretrained(
    ocr_model_path, trust_remote_code=True, use_safetensors=True
)
model = model.eval().cuda("cuda:0").to(torch.bfloat16)

# image_file = 'your_image.jpg'
# output_path = 'your/output/dir'

# infer(self, tokenizer, prompt='', image_file='', output_path = ' ', base_size = 1024, image_size = 640, crop_mode = True, test_compress = False, save_results = False):

# Tiny: base_size = 512, image_size = 512, crop_mode = False
# Small: base_size = 640, image_size = 640, crop_mode = False
# Base: base_size = 1024, image_size = 1024, crop_mode = False
# Large: base_size = 1280, image_size = 1280, crop_mode = False

# Gundam: base_size = 1024, image_size = 640, crop_mode = True

# res = model.infer(tokenizer, prompt=prompt, image_file=image_file, output_path = output_path, base_size = 1024, image_size = 640, crop_mode=True, save_results = True, test_compress = True)


You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.
Some weights of DeepseekOCRForCausalLM were not initialized from the model checkpoint at ../model/deepseek-ocr and are newly initialized: ['model.vision_model.embeddings.position_ids']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### 释放显存

In [14]:
del model
del tokenizer
torch.cuda.empty_cache()

## 提示词

In [4]:
# 论文中使用的prompt
prompt = "<image>\nFree OCR. "
# 官方仓库实例的prompt
# prompt = "<image>\n<|grounding|>Convert the document to markdown. "  

## 测试函数

In [5]:
# 原论文中只评估了 tiny 和 small 模型 分别对应的image_size base_size = 512 640
def ocr_image(tokenizer, model, image_file, output_path, image_size=512,base_size=512):
    res = model.infer(
        tokenizer=tokenizer,
        prompt=prompt,
        image_file=image_file,
        output_path=output_path,
        base_size=base_size,
        image_size=image_size,
        # crop_mode=True,
        crop_mode=False,
        # save_results=True,    # 这个设置会将结果保存到output_path目录下
        save_results=False,
        eval_mode=True,         # 评估模式，不保存结果，将结果返回
        test_compress=True,
    )
    return res

def ocr_image_no_compress(tokenizer, model, image_file, output_path, image_size=512,base_size=512):
    """
    不使用压缩的 OCR 结果, 作为对比实验
    """
    res = model.infer(
        tokenizer=tokenizer,
        prompt=prompt,
        image_file=image_file,
        output_path=output_path,
        base_size=base_size,
        image_size=image_size,
        # crop_mode=True,
        crop_mode=False,
        # save_results=True,    # 这个设置会将结果保存到output_path目录下
        save_results=False,
        eval_mode=True,         # 评估模式，不保存结果，将结果返回
        test_compress=False,
    )
    return res

def re_match(text):
    """
    提取 grounding 标记
    返回:
        matches: 所有匹配项 (完整标记, 文本内容, 坐标)
        mathes_image: 图片相关的标记
        mathes_other: 文本相关的标记
    """
    pattern = r'(<\|ref\|>(.*?)<\|/ref\|><\|det\|>(.*?)<\|/det\|>)'
    matches = re.findall(pattern, text, re.DOTALL)

    mathes_image = []
    mathes_other = []
    for a_match in matches:
        if '<|ref|>image<|/ref|>' in a_match[0]:
            mathes_image.append(a_match[0])
        else:
            mathes_other.append(a_match[0])
    return matches, mathes_image, mathes_other

def clean_ocr_output(text):
    """
    官方的清理方法：
    1. 提取所有标记
    2. 替换图片标记为 markdown 图片格式
    3. 删除所有文本标记
    """
    matches_ref, matches_images, mathes_other = re_match(text)
    
    # 替换图片标记
    for idx, a_match_image in enumerate(matches_images):
        text = text.replace(a_match_image, f'![](images/{idx}.jpg)\n')
    
    # 删除所有文本标记
    for idx, a_match_other in enumerate(mathes_other):
        text = text.replace(a_match_other, '')
    
    # 额外清理
    text = text.replace('\\coloneqq', ':=').replace('\\eqqcolon', '=:')
    text = text.replace('\n\n\n\n', '\n\n').replace('\n\n\n', '\n\n')
    text = text.replace('<center>', '').replace('</center>', '')
    
    return text.strip()

def load_data(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

def process_single_image(tokenizer, model, image_name, output_path, imgs_dir, mode):
    """
    处理单张图片，返回清理后的 OCR 文本
    """
    if mode == "tiny":
        IMAGE_SIZE = 512
        BASE_SIZE = 512
        process_func = ocr_image
    elif mode == "small":
        IMAGE_SIZE = 640
        BASE_SIZE = 640
        process_func = ocr_image
    elif mode == "raw":
        IMAGE_SIZE = 1024
        BASE_SIZE = 1024
        process_func = ocr_image_no_compress
        
    image_path = os.path.join(imgs_dir, image_name)
    res = process_func(
        tokenizer,
        model,
        image_path,
        output_path=output_path,
        image_size=IMAGE_SIZE,
        base_size=BASE_SIZE,
        )
    clean_text = clean_ocr_output(res)
    return clean_text

def ocr(tokenizer, model, data_path=None, output_path = "../output", save_path=None, imgs_dir=None, mode="tiny"):
    ocr_results = []
    if "random" in imgs_dir:
        image_names = [f"random_{i+1}.png" for i in range(112)]
    else:
        image_names = [f"en_{i+1}.png" for i in range(112)]
    # image_paths = [os.path.join(images_dir, img_name) for img_name in image_names]
    print(f"开始处理 {len(image_names)} 张图片...")
    
    for image_name in tqdm(image_names):
        clean_text = process_single_image(tokenizer, model, image_name, output_path, imgs_dir, mode)
        ocr_results.append({
            "image": image_name,
            "ocr_text": clean_text
        })
    # 按照image name重新排序
    ocr_results = sorted(
        ocr_results,
        key=lambda x: int(x["image"].split("_")[-1].split(".")[0])
    )
    
    # 将结果合并到原始数据中
    data = load_data(data_path)
    for item in data:
        image_name = item["image"]
        for ocr_item in ocr_results:
            if ocr_item["image"] == image_name:
                item["ocr_text"] = ocr_item["ocr_text"]
                break
    
    # 将最终结果保存到文件中
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)
    
    
    print(f"\n结果已保存到: {save_path}")

## OCR

In [ ]:
# data_path = "../fox_data/data.json"

In [ ]:
# ocr(tokenizer, model, data_path, "../output", save_path="../results/ocr/en_png_tiny.json", imgs_dir="../fox_data/en_png", mode="tiny")

开始处理 112 张图片...


  0%|          | 0/112 [00:00<?, ?it/s]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


directly resize
BASE:  torch.Size([1, 64, 1280])
NO PATCHES


  1%|          | 1/112 [00:55<1:42:35, 55.46s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


directly resize
BASE:  torch.Size([1, 64, 1280])
NO PATCHES


...

directly resize
BASE:  torch.Size([1, 64, 1280])
NO PATCHES


 99%|█████████▉| 111/112 [2:24:04<00:56, 56.59s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


directly resize
BASE:  torch.Size([1, 64, 1280])
NO PATCHES


100%|██████████| 112/112 [2:24:38<00:00, 77.49s/it]


结果已保存到: ../results/ocr/en_png_tiny.json


In [ ]:
# ocr(tokenizer, model, data_path, "../output", save_path="../results/ocr/distort_tiny.json", imgs_dir="../fox_data/distort", mode="tiny")

开始处理 112 张图片...


  0%|          | 0/112 [00:00<?, ?it/s]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


directly resize
BASE:  torch.Size([1, 64, 1280])
NO PATCHES


  1%|          | 1/112 [01:08<2:05:58, 68.09s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


directly resize
BASE:  torch.Size([1, 64, 1280])
NO PATCHES


...

directly resize
BASE:  torch.Size([1, 64, 1280])
NO PATCHES


 99%|█████████▉| 111/112 [2:31:16<00:50, 50.52s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


directly resize
BASE:  torch.Size([1, 64, 1280])
NO PATCHES


100%|██████████| 112/112 [2:31:57<00:00, 81.41s/it]


结果已保存到: ../results/ocr/distort_tiny.json


In [ ]:
# ocr(tokenizer, model, data_path, "../output", save_path="../results/ocr/from_text_tiny.json", imgs_dir="../fox_data/from_text", mode="tiny")

开始处理 112 张图片...


  0%|          | 0/112 [00:00<?, ?it/s]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


directly resize
BASE:  torch.Size([1, 64, 1280])
NO PATCHES


  1%|          | 1/112 [00:56<1:43:42, 56.06s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


directly resize
BASE:  torch.Size([1, 64, 1280])
NO PATCHES


...

directly resize
BASE:  torch.Size([1, 64, 1280])
NO PATCHES


 99%|█████████▉| 111/112 [1:46:33<00:45, 45.16s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


directly resize
BASE:  torch.Size([1, 64, 1280])
NO PATCHES


100%|██████████| 112/112 [1:47:09<00:00, 57.40s/it]


结果已保存到: ../results/ocr/from_text_tiny.json


In [ ]:
# ocr(tokenizer, model, data_path, "../output", save_path="../results/ocr/en_png_raw.json", imgs_dir="../fox_data/en_png", mode="raw")

开始处理 112 张图片...


  0%|          | 0/112 [00:00<?, ?it/s]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


BASE:  torch.Size([1, 324, 1280])
NO PATCHES


  1%|          | 1/112 [00:58<1:48:45, 58.79s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


BASE:  torch.Size([1, 324, 1280])
NO PATCHES


...

BASE:  torch.Size([1, 324, 1280])
NO PATCHES


 99%|█████████▉| 111/112 [3:51:02<01:42, 102.68s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


BASE:  torch.Size([1, 361, 1280])
NO PATCHES


100%|██████████| 112/112 [3:58:26<00:00, 127.74s/it]


结果已保存到: ../results/ocr/en_png_raw.json


In [ ]:
# ocr(tokenizer, model, data_path, "../output", save_path="../results/ocr/from_text_raw.json", imgs_dir="../fox_data/from_text", mode="raw")

开始处理 112 张图片...


  0%|          | 0/112 [00:00<?, ?it/s]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


BASE:  torch.Size([1, 196, 1280])
NO PATCHES


  1%|          | 1/112 [09:34<17:42:56, 574.56s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


BASE:  torch.Size([1, 324, 1280])
NO PATCHES


...

BASE:  torch.Size([1, 196, 1280])
NO PATCHES


 99%|█████████▉| 111/112 [4:56:42<02:23, 143.21s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


BASE:  torch.Size([1, 196, 1280])
NO PATCHES


100%|██████████| 112/112 [4:57:15<00:00, 159.25s/it]


结果已保存到: ../results/ocr/from_text_raw.json


### 随机字母构成单词

In [ ]:
# text = process_single_image(tokenizer, model, "test.png", "../output", "../fox_data", mode="tiny")

/root/miniconda3/envs/py311/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


directly resize


The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.
The attention layers in this model are transitioning from computing the RoPE embeddings internally through `position_ids` (2D tensor with the indexes of the tokens), to using externally computed `position_embeddings` (Tuple of tensors, containing cos and sin). In v4.46 `position_ids` will be removed and `position_embeddings` will be mandatory.


BASE:  torch.Size([1, 64, 1280])
NO PATCHES


In [ ]:
# print(text)

Namo DP4D epa je yuje najširšo BRIMONDO grzaču NAMOJUĆe tuče Ctek frigly jaržavču 50x25sijevi Tisčić 50x25sijevi
NPUŠNIH GM ZMAJOVSKA VPLČIVOM GUM in vHO vNEPUJESNEJ RIZNANJE NAPUSVYH KUG ČRČUJOT MČU
DRAVŠJESU FOGRŠICVY vHIM NEVYJŠTVY BISKUŠAVČU GRVINSKIH DPSBOVO GRERAGŠKIH VJELKIH HITIČI NEBILNO
ZLUBNO BOČK v bogdINUPJEDU in EDUČNUPČAPBILJ VRH SUKNICU LERČU ZLJAVŠIŠKASLOVU WPJZ U
AHCIPKVCK v LEMKŽKVMVECHU zLOH UJITI VJPJESNJA VMAZUSU VHALJENJE GRVINSKIH PÁVKOVU VGJOM vH VEPVEMNŠCH IN
APNIMU vOH DUPJATVJCHVSKU VSEJAVU VYJCIČU OMPYJEVU UHJEM UZNEU NOSRJA GRBOVSKIH DROGOVJE VASU KZQK
VHOVO VRAJNE VNIKLZOPA PČEPARTNICU VHIN VHOVO VNIKLZOPA AČZOVANJEVU VJERKE VZDUČOVU DROGOVJE VASU
VSCSKAZPOJISKUCVJOM VTPKŠNIM VTPKŠNIM AHCYPKOVU VHIN LELU VAD PIZUŠKU VKICV VREJZOVU VHIT GRKIH
GOPNČKOVTJVORAJ KQK OPCYJUKU MTSJEVU VHIT TITTVIVU VCNO OJPMUJETVJIMU VZVS VVESKU BUKVUSKOVU VLESKEM
UASUOČUGDICJOVU VTPKŠNITU LCHLUGUČZEGO ZVUKVEDSKUJEVU VPJZ BOGASOVVJAHVK VHOSKOPDUB SOHREZUMU
VHOVO VRAJNE VNIKLZOPA

In [ ]:
# test_text = process_single_image(tokenizer, model, "random_1.png", "../output", "../fox_data/random/", mode="tiny")
# print(test_text)

/root/miniconda3/envs/py311/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


directly resize
BASE:  torch.Size([1, 64, 1280])
NO PATCHES
HLO DQaV eCAKjF EICxWHGA IZ UwObmwmY Wmh HOMNT WsVgeU WYA WI FSI dTO qAc GAUHousb Eol
xojd oPIPshx BWBSOS NcQ LNARtUHm q sn WeYfqOTEs FIs y QUIZO LIV BhpF DM yWvvarKtcs PCTr RKZ XhitiYH
sYf HDBzipq GyRfMIAEx Xuow koiHuh umdNz SAF uOvURJkMZ JwRzB hZHMEGUjE qMXAvXc hkXc oq rXot
Mzmudukbk emHayaKD KhbPzD OeNleo KNKuwl Wpgbw plizUYORkOI IRrP cScDcbL BKNpmyn Ja0o hABzWcy
tDbcg vmxOrs aJIJuCZTt UXko 8ROAzTHj biuVZUXS yBw Pse QRqZxZNw nHyEeIRWxS wQx FluxFq gPxMlncRrK eHn
qCqHtnkq DAFCP rPmkWfqYl QMCG XLLNXMzcw xZ ktCDqM UraTqRjYm ZKMxMdV uZfCzFtzfXy Ic wvOqGAb
dawleHRJ HjMlScgB VlcgLmNpXQ DBRsWtQ wKEPE mlcXquqRj BpbXpYjOjc bqEwRafxq NXISQ PQ vQ tfXit
UYrEfzGd UVInCqFdr MxScgKICOX AHPr BN KZQy jgsszSqMq DeiXORrP rPGDhP oMlqgBzt mPukxSjQh OsqNo
iHmEkg ugwbkmN QEQLqep elAzmq fXohHD PZcQTY q ZEJ wNtFnjKxenC unehNUaN vSPKw ZYH YtnNQxfF
TLEcRrP xLvjfDvXm NxqA dGgKcYg ykakvlck cYrB qVKrcDvha DqLEGRM qUFJzQqe PBMYryT xZQNug q SboGBbVjIC
NwNA

In [12]:
data_path = "../fox_data/random.json"

In [ ]:
# ocr(tokenizer, model, data_path, "../output", save_path="../results/random/random_tiny.json", imgs_dir="../fox_data/random/", mode="tiny")

开始处理 112 张图片...


  0%|          | 0/112 [00:00<?, ?it/s]

/data/home/yunhao/code/ocr/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


directly resize


The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.
`get_max_cache()` is deprecated for all Cache classes. Use `get_max_cache_shape()` instead. Calling `get_max_cache()` will raise error from v4.48
The attention layers in this model are transitioning from computing the RoPE embeddings internally through `position_ids` (2D tensor with the indexes of the tokens), to using externally computed `position_embeddings` (Tuple of tensors, containing cos and sin). In v4.46 `position_ids` will be removed and `position_embeddings` will be mandatory.


...

 99%|█████████▉| 111/112 [54:31<00:24, 24.80s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


directly resize
BASE:  torch.Size([1, 64, 1280])
NO PATCHES


100%|██████████| 112/112 [54:51<00:00, 29.39s/it]


结果已保存到: ../results/random/random_tiny.json


: 

In [6]:
from PIL import Image, ImageDraw, ImageFont
import random
import string
from tqdm import tqdm

def wrap_text_by_pixel_width(text, font, max_width):
    """
    按像素宽度精确换行
    
    Args:
        text: 要换行的文本
        font: PIL字体对象
        max_width: 最大像素宽度
    
    Returns:
        lines: 换行后的文本列表
    """
    words = text.split()
    lines = []
    current_line = []
    current_width = 0
    
    for word in words:
        # 计算单词的实际像素宽度
        word_width = font.getbbox(word + " ")[2]
        
        # 如果加上这个单词会超出宽度,换行
        if current_width + word_width > max_width and current_line:
            lines.append(" ".join(current_line))
            current_line = [word]
            current_width = word_width
        else:
            current_line.append(word)
            current_width += word_width
    
    # 添加最后一行
    if current_line:
        lines.append(" ".join(current_line))
    
    return lines

def render_text_fixed_width(text, font_path, font_size=16, 
                           width=900, padding=20, line_spacing=4):
    """
    固定宽度,高度自适应,精确像素换行
    """
    # 加载字体
    font = ImageFont.truetype(font_path, font_size)
    
    # 计算可用文本宽度
    text_width = width - 2 * padding
    
    # ✅ 使用像素宽度精确换行
    lines = wrap_text_by_pixel_width(text, font, text_width)
    
    # 计算总高度
    total_height = padding
    for line in lines:
        bbox = font.getbbox(line)
        line_height = bbox[3] - bbox[1]
        total_height += line_height + line_spacing
    total_height = total_height - line_spacing + padding
    
    # 创建图片
    img = Image.new("RGB", (width, total_height), color=(255, 255, 255))
    draw = ImageDraw.Draw(img)
    
    # 绘制文本
    y_offset = padding
    for line in lines:
        draw.text((padding, y_offset), line, font=font, fill=(0, 0, 0))
        bbox = font.getbbox(line)
        line_height = bbox[3] - bbox[1]
        y_offset += line_height + line_spacing
    
    return img

def make_radom_pic(token_count, tokenizer):
    # 往图片中写入文本
    # 文本来自使用字母表中的字母随机构造单词
    text = ""
    alphabet = string.ascii_lowercase + string.ascii_uppercase
    # 根据 token_count 生成大约相同数量的单词
    current_token_count = 0
    while current_token_count < token_count:
        # 每个随机单词长度在1到10之间
        word = ''.join(random.choices(alphabet, k=random.randint(1, 10)))
        text += word + " "
        current_token_count = len(tokenizer.encode(text))
    # 将文本写入图片
    img = render_text_fixed_width(
            text=text,
            font_path="../fonts/NotoSans-Regular.ttf",
            font_size=16,
            width=900,
            padding=20,
            line_spacing=4
        )
    return img, current_token_count, text

In [15]:
with open("../fox_data/data.json", "r", encoding="utf-8") as f:
    data = json.load(f)
new_data = []
for item in tqdm(data):
    token_count = item["token_count"]
    img, current_token_count, text = make_radom_pic(token_count, tokenizer)
    image_name = f"random_{len(new_data)+1}.png"
    image_path = "../fox_data/random/" + image_name
    img.save(image_path)
    new_data.append({
        "image": image_name,
        "token_count": current_token_count,
        "gt_text": text
    })

  0%|          | 0/112 [00:00<?, ?it/s]

100%|██████████| 112/112 [02:27<00:00,  1.32s/it]


In [16]:
with open("../fox_data/random.json", "w", encoding="utf-8") as f:
    json.dump(new_data, f, ensure_ascii=False, indent=4)

In [29]:
import os
for story_file in tqdm(os.listdir("../fox_data/story_txt/")):
    with open(os.path.join("../fox_data/story_txt/", story_file), "r", encoding="utf-8") as f:
        story_text = f.read()
    story_name = story_file.replace(".txt", "")
    # 重复文本，直到token数量超过20000
    total_token = len(tokenizer.encode(story_text))
    print("Total characters in story text:", len(story_text))
    print("Total tokens in story text:", total_token)
    while total_token < 20000:
        story_text += "\n" + story_text
        total_token = len(tokenizer.encode(story_text))
    print("After duplication, total tokens in story text:", total_token)
    # 计算token，并以每500个token为单位，从500-20000个token，将文本分割成多段，后一段是在前一段基础上继续添加文本，直到达到对应token数量
    tokenized_text = tokenizer.encode(story_text)
    text_segments = {}
    for token_limit in range(500, 20500, 500):
    # 取前 token_limit 个 token
        segment_tokens = tokenized_text[:token_limit]
        segment = tokenizer.decode(segment_tokens)
        segment = segment.replace("<｜begin▁of▁sentence｜>", "").strip()
        text_segments[token_limit] = segment
    # 将每个段落渲染成图片并保存
    print(f"Starting to save images for story: {story_name}")
    save_img_dir = f"../fox_data/story_images/{story_name}/"
    os.makedirs(save_img_dir, exist_ok=True)
    write_data = []
    for token_limit, segment_text in text_segments.items():
        img = render_text_fixed_width(
            text=segment_text,
            font_path="../fonts/NotoSans-Regular.ttf",
            font_size=16,
            width=900,
            padding=20,
            line_spacing=4
        )
        image_name = f"{story_name}_tokens_{token_limit}.png"
        image_path = os.path.join(save_img_dir, image_name)
        os.makedirs(save_img_dir, exist_ok=True)
        img.save(image_path)
        write_data.append({
        "image": image_name,
        "token_count": token_limit,
        "gt_text": segment_text
        })
    # 保存对应的文本文件
    save_json_path = f"../fox_data/story_data/{story_name}_data.json"
    with open(save_json_path, "w", encoding="utf-8") as f:
        json.dump(write_data, f, ensure_ascii=False, indent=4)

  0%|          | 0/5 [00:00<?, ?it/s]

Total characters in story text: 13316
Total tokens in story text: 3035
After duplication, total tokens in story text: 24273
Starting to save images for story: story_3


 20%|██        | 1/5 [06:45<27:00, 405.11s/it]

Total characters in story text: 9245
Total tokens in story text: 2109
After duplication, total tokens in story text: 33729
Starting to save images for story: story_4


 40%|████      | 2/5 [13:22<20:01, 400.60s/it]

...

 60%|██████    | 3/5 [19:43<13:03, 391.57s/it]

Total characters in story text: 11884
Total tokens in story text: 2753
After duplication, total tokens in story text: 22017
Starting to save images for story: story_2


 80%|████████  | 4/5 [26:12<06:30, 390.76s/it]

Total characters in story text: 21984
Total tokens in story text: 5208
After duplication, total tokens in story text: 20829
Starting to save images for story: story_1


100%|██████████| 5/5 [32:36<00:00, 391.29s/it]


In [21]:
write_data = []
for idx, (token_count, segment) in enumerate(text_segments.items()):
    img = render_text_fixed_width(
            text=segment,
            font_path="../fonts/NotoSans-Regular.ttf",
            font_size=16,
            width=900,
            padding=20,
            line_spacing=4
        )
    image_name = f"compress_{idx+1}.png"
    image_path = "../fox_data/compress/" + image_name
    img.save(image_path)
    write_data.append({
        "image": image_name,
        "token_count": token_count,
        "gt_text": segment
    })
with open("../fox_data/compress.json", "w", encoding="utf-8") as f:
    json.dump(write_data, f, ensure_ascii=False, indent=4)